# Flash-VStream Ragged Memory Pipeline (Colab)

这个 notebook 用于在 Google Colab 运行你当前的 Route-B Ragged pipeline：
- `retriever-only`（快速验证预算与检索）
- `model-wrapped`（模型包装路径验证）

默认会检查 `N_sel <= 12000`，并输出每 chunk 的日志与汇总。

In [ ]:
import os
from pathlib import Path

# 方式1：你已经把仓库上传到 Colab /content 下
REPO_DIR = Path('/content/Flash-VStream-highres')

# 方式2：如果你希望直接 git clone（把 URL 改成你的仓库地址）
REPO_URL = ''  # 例如: 'https://github.com/<you>/Flash-VStream-highres.git'

if not REPO_DIR.exists():
    if REPO_URL.strip():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        raise FileNotFoundError('未找到 /content/Flash-VStream-highres，请先上传仓库或填写 REPO_URL')

%cd {REPO_DIR}
print('Repo ready:', REPO_DIR)

In [ ]:
# Colab 通常已带 torch；这里补齐常见依赖
!pip -q install -U transformers accelerate sentencepiece einops

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import importlib.util

required = [
    'pipeline_test_ragged_streaming_dummy.py',
    'Flash-VStream-Qwen-highres/models/ragged_flash_memory_retriever.py',
    'Flash-VStream-Qwen-highres/models/flash_memory_constants.py',
]
for p in required:
    fp = Path(p)
    print(f'{p}:', 'OK' if fp.exists() else 'MISSING')

spec = importlib.util.spec_from_file_location('flash_consts', 'Flash-VStream-Qwen-highres/models/flash_memory_constants.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print('DEFAULT_FLASH_MEMORY_CONFIG =', mod.DEFAULT_FLASH_MEMORY_CONFIG)

## 1) Retriever-only 快速测试
建议先跑这个，确认 ragged 检索和预算控制逻辑稳定。

In [ ]:
!python pipeline_test_ragged_streaming_dummy.py \
  --mode retriever-only \
  --num_chunks 10 \
  --budget_target 11520 \
  --budget_hard 12000 \
  --r_t 0.3333 \
  --num_anchors 8 \
  --topM_items 10 \
  --log_level INFO

## 2) Model-wrapped 测试
这一步会走 `embed_new_video_clip` + `prepare_realtime_inference` 包装路径，并做 save/reload dummy 配置文件。

In [ ]:
!python pipeline_test_ragged_streaming_dummy.py \
  --mode model-wrapped \
  --num_chunks 10 \
  --frames_per_chunk 8 \
  --budget_target 11520 \
  --budget_hard 12000 \
  --r_t 0.3333 \
  --num_anchors 8 \
  --topM_items 10 \
  --log_level INFO

In [ ]:
import json
from pathlib import Path

cfg = Path('tmp_ragged_dummy_ckpt/ragged_config.json')
if cfg.exists():
    data = json.loads(cfg.read_text())
    print('Reloaded dummy ragged config:', data)
    assert data.get('flash_memory_budget_hard', 0) <= 12000
else:
    print('No dummy ckpt found (check previous cell logs).')

## 说明
- 如果你后续接真实模型权重，只需要把 `pipeline_test_ragged_streaming_dummy.py` 的 dummy visual/model 路径替换为真实加载路径。
- 当前 notebook 已覆盖你要求的核心验证：动态分辨率 chunk、预算约束（<=12000）、每步日志、汇总输出。